# Predictive Maintenance - LSTM & Final Evaluation

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import time

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import shap

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print(f" PyTorch version: {torch.__version__}")
print(f" CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
else:
    print("   Using CPU")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n Using device: {device}")

 PyTorch version: 2.1.2+cpu
 CUDA available: False
   Using CPU

 Using device: cpu


### 1. Load Data

In [3]:
train_processed = pd.read_csv('../data/processed/train_processed.csv')

print("Data Loading")
print(f"\n Dataset shape: {train_processed.shape}")

feature_cols = [col for col in train_processed.columns if col not in ['unit_id', 'RUL']]
X = train_processed[feature_cols].values
y = train_processed['RUL'].values
unit_ids = train_processed['unit_id'].values

print(f" Number of features: {len(feature_cols)}")
print(f" Target variable: RUL")

Data Loading

 Dataset shape: (20631, 113)
 Number of features: 111
 Target variable: RUL


### 2. Prepare Sequence for LSTM

    Create sequences for LSTM.
    
    Args:
        X: Feature array
        y: Target array
        unit_ids: Engine IDs
        time_steps: Sequence length
    
    Returns:
        X_seq: Sequences of shape (n_sequences, time_steps, n_features)
        y_seq: Targets of shape (n_sequences,)
        engine_ids_seq: Engine IDs for each sequence

In [4]:
def create_sequences(X, y, unit_ids, time_steps=10):
    X_seq, y_seq, engine_ids_seq = [], [], []

    for engine_id in np.unique(unit_ids):
        engine_mask = unit_ids == engine_id
        X_engine = X[engine_mask]
        y_engine = y[engine_mask]

        for i in range(len(X_engine) - time_steps):
            X_seq.append(X_engine[i:(i + time_steps)])
            y_seq.append(y_engine[i + time_steps])
            engine_ids_seq.append(engine_id)
    
    return np.array(X_seq), np.array(y_seq), np.array(engine_ids_seq)

print("Creating Sequences For LSTM")

TIME_STEPS = 10
print(f"\n Sequence window size: {TIME_STEPS} time steps")
print("\n Creating sequences... ")

start_time = time.time()
X_seq, y_seq, engine_ids_seq = create_sequences(X, y, unit_ids, time_steps=TIME_STEPS)
creation_time = time.time() - start_time

print(f"\n Sequences created in {creation_time:.2f} seconds")
print(f"\n Sequence shapes:")
print(f"   - X_seq: {X_seq.shape} (n_sequences, time_steps, n_features)")
print(f"   - y_seq: {y_seq.shape} (n_sequences,)")
print(f"\n Explanation:")
print(f"   - We have {X_seq.shape[0]} sequences")
print(f"   - Each sequence has {X_seq.shape[1]} time steps")
print(f"   - Each time step has {X_seq.shape[2]} features")

Creating Sequences For LSTM

 Sequence window size: 10 time steps

 Creating sequences... 

 Sequences created in 0.39 seconds

 Sequence shapes:
   - X_seq: (19631, 10, 111) (n_sequences, time_steps, n_features)
   - y_seq: (19631,) (n_sequences,)

 Explanation:
   - We have 19631 sequences
   - Each sequence has 10 time steps
   - Each time step has 111 features


### 3. Train-Test Split for LSTM

In [5]:
unique_engines = np.unique(engine_ids_seq)
n_train_engines = int(len(unique_engines) * 0.8)

train_engines = unique_engines[:n_train_engines]
test_engines = unique_engines[n_train_engines:]

train_mask = np.isin(engine_ids_seq, train_engines)
test_mask = np.isin(engine_ids_seq, test_engines)

X_train = X_seq[train_mask]
X_test = X_seq[test_mask]
y_train = y_seq[train_mask]
y_test = y_seq[test_mask]

print("Train-Test Split")
print(f"\n Split complete:")
print(f"   - Training sequences: {len(X_train)}")
print(f"   - Testing sequences: {len(X_test)}")
print(f"   - Training engines: {len(train_engines)}")
print(f"   - Testing engines: {len(test_engines)}")

overlap = set(train_engines) & set(test_engines)
print(f"\n Data leakage check: {' PASSED' if not overlap else ' FAILED'}")

Train-Test Split

 Split complete:
   - Training sequences: 15338
   - Testing sequences: 4293
   - Training engines: 80
   - Testing engines: 20

 Data leakage check:  PASSED
